In [1]:
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
ENV_NAME = "☁️ Google Colab" if IN_COLAB else "💻 Ambiente Local (WSL/Jupyter)"

print(f"Detectado: {ENV_NAME}")
print(f"Versão do Python: {sys.version.split()[0]}")

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = "/content/drive/MyDrive/fiap/segundo ano/challenge_locaweb/2_silver_data/"
    SAVE_PATH = "/content/drive/MyDrive/fiap/segundo ano/challenge_locaweb/3_golden_data/"
else:
    BASE_PATH = "../2_silver_data/"
    SAVE_PATH = "../3_golden_data/"

Detectado: ☁️ Google Colab
Versão do Python: 3.12.13
Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow.dataset as ds
from scipy import stats

### Importação de features

In [8]:
s_diario_volume_abertura = pd.read_csv(filepath_or_buffer=f"{BASE_PATH}s_diario_volume_abertura.csv", sep=";", encoding='ISO-8859-1', parse_dates=['data'])
s_diario_volume_abertura.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1095 entries, 0 to 1094
Data columns (total 17 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   data                      1095 non-null   datetime64[ns]
 1   inc_abertos               1095 non-null   int64         
 2   inc_abertos_p1            1095 non-null   int64         
 3   inc_abertos_p2            1095 non-null   int64         
 4   inc_abertos_p3            1095 non-null   int64         
 5   inc_abertos_p4            1095 non-null   int64         
 6   inc_abertos_p5            1095 non-null   int64         
 7   inc_abertos_ac_mes        1095 non-null   int64         
 8   inc_abertos_ac_ano        1095 non-null   int64         
 9   inc_abertos_ac_30d        1095 non-null   int64         
 10  inc_abertos_ac_7d         1095 non-null   int64         
 11  inc_abertos_madrugada     1095 non-null   int64         
 12  inc_abertos_manha   

In [9]:
s_diario_temporal = pd.read_csv(filepath_or_buffer=f"{BASE_PATH}s_diario_temporal.csv", sep=";", encoding='ISO-8859-1', parse_dates=['data'])
s_diario_temporal.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1095 entries, 0 to 1094
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   data          1095 non-null   datetime64[ns]
 1   feriado       1095 non-null   int64         
 2   nome_feriado  44 non-null     object        
 3   dia_semana    1095 non-null   int64         
 4   dia_util      1095 non-null   int64         
 5   mes           1095 non-null   int64         
 6   trimestre     1095 non-null   int64         
 7   dia_mes       1095 non-null   int64         
 8   num_dia_util  1095 non-null   int64         
 9   ano_mes       1095 non-null   object        
dtypes: datetime64[ns](1), int64(7), object(2)
memory usage: 85.7+ KB


In [11]:
s_diario_monitoramento = pd.read_csv(filepath_or_buffer=f"{BASE_PATH}s_diario_monitoramento.csv", sep=";", encoding='ISO-8859-1', parse_dates=['data'])
s_diario_monitoramento.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1095 entries, 0 to 1094
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   data               1095 non-null   datetime64[ns]
 1   inc                1095 non-null   int64         
 2   inc_mon            1095 non-null   int64         
 3   taxa_mon_diario    1095 non-null   float64       
 4   inc_ac_mes         1095 non-null   int64         
 5   inc_ac_mon_mes     1095 non-null   int64         
 6   taxa_mon_ac_mes    1095 non-null   float64       
 7   inc_ac_geral       1095 non-null   int64         
 8   inc_ac_mon_geral   1095 non-null   int64         
 9   taxa_mon_ac_geral  1095 non-null   float64       
dtypes: datetime64[ns](1), float64(3), int64(6)
memory usage: 85.7 KB


In [12]:
s_diario_itens_config = pd.read_csv(filepath_or_buffer=f"{BASE_PATH}s_diario_itens_config.csv", sep=";", encoding='ISO-8859-1', parse_dates=['data'])
s_diario_itens_config.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1095 entries, 0 to 1094
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   data                 1095 non-null   datetime64[ns]
 1   ics_outliers_ativos  1095 non-null   int64         
 2   ics_inativos         1095 non-null   int64         
 3   ics_total            1095 non-null   int64         
dtypes: datetime64[ns](1), int64(3)
memory usage: 34.3 KB


In [10]:
s_diario_volume_fechamento = pd.read_csv(filepath_or_buffer=f"{BASE_PATH}s_diario_volume_fechamento.csv", sep=";", encoding='ISO-8859-1', parse_dates=['data'])
s_diario_volume_fechamento.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1095 entries, 0 to 1094
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   data                       1095 non-null   datetime64[ns]
 1   inc                        1095 non-null   float64       
 2   inc_fechados_p1            1095 non-null   float64       
 3   inc_fechados_p2            1095 non-null   float64       
 4   inc_fechados_p3            1095 non-null   float64       
 5   inc_fechados_p4            1095 non-null   float64       
 6   inc_fechados_p5            1095 non-null   float64       
 7   inc_fechados_ac_mes        1095 non-null   float64       
 8   inc_fechados_ac_ano        1095 non-null   float64       
 9   inc_fechados_ac_30d        1095 non-null   float64       
 10  inc_fechados_ac_7d         1095 non-null   float64       
 11  inc_p2_fechados_ac_mes     1095 non-null   float64       
 12  inc_p3

In [13]:
s_diario_status_fechamento = pd.read_csv(filepath_or_buffer=f"{BASE_PATH}s_diario_status_fechamento.csv", sep=";", encoding='ISO-8859-1', parse_dates=['data'])
s_diario_status_fechamento.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1095 entries, 0 to 1094
Data columns (total 29 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   data                       1095 non-null   datetime64[ns]
 1   total_incidentes_fechados  1095 non-null   int64         
 2   inc_stat_ea                1095 non-null   int64         
 3   inc_stat_e                 1095 non-null   int64         
 4   inc_stat_si                1095 non-null   int64         
 5   taxa_stat_ea_diario        1095 non-null   float64       
 6   taxa_stat_e_diario         1095 non-null   float64       
 7   taxa_stat_si_diario        1095 non-null   float64       
 8   inc_fechados_ac_mes        1095 non-null   float64       
 9   inc_ac_stat_ea_mes         1095 non-null   int64         
 10  inc_ac_stat_e_mes          1095 non-null   int64         
 11  inc_ac_stat_si_mes         1095 non-null   int64         
 12  taxa_s

Data Processing

In [17]:
training_table = pd.melt(
    s_diario_volume_abertura,
    id_vars=['data'], # A coluna que deve continuar intacta/fixa
    value_vars=['inc_abertos_p1', 'inc_abertos_p2', 'inc_abertos_p3', 'inc_abertos_p4', 'inc_abertos_p5'], # As colunas que vão virar variáveis
    var_name='prioridade', # O nome da nova coluna que vai receber os nomes (inc_p1, etc.)
    value_name='qtd_inc' # O nome da nova coluna que vai receber os valores numéricos
)
training_table['prioridade'] = training_table['prioridade'].str.replace('inc_abertos_p', '').astype(int)
training_table.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5475 entries, 0 to 5474
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   data        5475 non-null   datetime64[ns]
 1   prioridade  5475 non-null   int64         
 2   qtd_inc     5475 non-null   int64         
dtypes: datetime64[ns](1), int64(2)
memory usage: 128.4 KB


In [18]:
training_table['dia_anterior'] = training_table['data'] - pd.Timedelta(days=1)
training_table.head()

,data,prioridade,qtd_inc,dia_anterior
0,2023-01-02,1,0,2023-01-01
1,2023-01-03,1,0,2023-01-02
2,2023-01-04,1,0,2023-01-03
3,2023-01-05,1,0,2023-01-04
4,2023-01-06,1,0,2023-01-05


In [19]:
training_table.sort_values(by=['data', 'prioridade'], inplace=True)
training_table.tail(20)

,data,prioridade,qtd_inc,dia_anterior
1091,2025-12-28,1,0,2025-12-27
2186,2025-12-28,2,46,2025-12-27
3281,2025-12-28,3,391,2025-12-27
4376,2025-12-28,4,392,2025-12-27
5471,2025-12-28,5,0,2025-12-27
1092,2025-12-29,1,0,2025-12-28
2187,2025-12-29,2,57,2025-12-28
3282,2025-12-29,3,515,2025-12-28
4377,2025-12-29,4,349,2025-12-28
5472,2025-12-29,5,0,2025-12-28


In [21]:
s_diario_volume_fechamento.columns

Index(['data', 'inc', 'inc_fechados_p1', 'inc_fechados_p2', 'inc_fechados_p3',
       'inc_fechados_p4', 'inc_fechados_p5', 'inc_fechados_ac_mes',
       'inc_fechados_ac_ano', 'inc_fechados_ac_30d', 'inc_fechados_ac_7d',
       'inc_p2_fechados_ac_mes', 'inc_p3_fechados_ac_mes',
       'inc_p2_fechados_ac_ano', 'inc_p3_fechados_ac_ano',
       'inc_fechados_madrugada', 'inc_fechados_manha', 'inc_fechados_tarde',
       'inc_fechados_noite', 'inc_fechados_hor_com',
       'inc_fechados_fora_hor_com'],
      dtype='object')

In [20]:
# unindo as features temporais - devem usar o dia real
training_table = training_table.merge(
    s_diario_temporal[['data', 'feriado', 'dia_semana', 'dia_util']],
    on='data',
    how='left'
)

# unindo as features de fechamento de incidentes - devem usar o dia anterior
# Correção: removida a duplicata de 'taxa_stat_e_diario' e aplicada a renomeação da chave
training_table = training_table.merge(
    s_diario_volume_fechamento[['data', 'inc_fechados_ac_7d']].rename(columns={'data': 'dia_anterior'}),
    on='dia_anterior',
    how='left'
)

# unindo as features de monitoramento - devem usar o dia anterior
training_table = training_table.merge(
    s_diario_monitoramento[['data', 'taxa_mon_diario']].rename(columns={'data': 'dia_anterior'}),
    on='dia_anterior',
    how='left'
)

# unindo as features relacionadas aos ics - devem usar o dia anterior
training_table = training_table.merge(
    s_diario_itens_config[['data', 'ics_outliers_ativos', 'ics_inativos']].rename(columns={'data': 'dia_anterior'}),
    on='dia_anterior',
    how='left'
)


KeyError: "['taxa_stat_si_diario', 'taxa_stat_ea_diario', 'taxa_stat_e_diario'] not in index"

In [ ]:
training_table.head(20)

,data,prioridade,qtd_inc,dia_anterior,feriado,dia_semana,dia_util,taxa_stat_si_diario,taxa_stat_ea_diario,taxa_stat_e_diario,taxa_mon_diario,ics_outliers_ativos,ics_inativos
0,2023-01-02,1,0,2023-01-01,0,0,1,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-01-02,2,0,2023-01-01,0,0,1,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-01-02,3,1,2023-01-01,0,0,1,NaN,NaN,NaN,NaN,NaN,NaN
3,2023-01-02,4,0,2023-01-01,0,0,1,NaN,NaN,NaN,NaN,NaN,NaN
4,2023-01-02,5,0,2023-01-01,0,0,1,NaN,NaN,NaN,NaN,NaN,NaN
5,2023-01-03,1,0,2023-01-02,0,1,1,0.0,0.0,0.0,0.0,0.0,1.0
6,2023-01-03,2,0,2023-01-02,0,1,1,0.0,0.0,0.0,0.0,0.0,1.0
7,2023-01-03,3,0,2023-01-02,0,1,1,0.0,0.0,0.0,0.0,0.0,1.0
8,2023-01-03,4,0,2023-01-02,0,1,1,0.0,0.0,0.0,0.0,0.0,1.0
9,2023-01-03,5,0,2023-01-02,0,1,1,0.0,0.0,0.0,0.0,0.0,1.0


In [ ]:
# Tabela final de treinamento
training_table_final = training_table.drop(columns=['dia_anterior'])
training_table_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5475 entries, 0 to 5474
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   data                 5475 non-null   datetime64[ns]
 1   prioridade           5475 non-null   int64         
 2   qtd_inc              5475 non-null   int64         
 3   feriado              5475 non-null   int64         
 4   dia_semana           5475 non-null   int64         
 5   dia_util             5475 non-null   int64         
 6   taxa_stat_si_diario  5470 non-null   float64       
 7   taxa_stat_ea_diario  5470 non-null   float64       
 8   taxa_stat_e_diario   5470 non-null   float64       
 9   taxa_mon_diario      5470 non-null   float64       
 10  ics_outliers_ativos  5470 non-null   float64       
 11  ics_inativos         5470 non-null   float64       
dtypes: datetime64[ns](1), float64(6), int64(5)
memory usage: 513.4 KB


In [ ]:
# Transformando a coluna data em index
training_table_final['data_prioridade'] = training_table_final['data'].astype(str) + '_p' + training_table_final['prioridade'].astype(str)
training_table_final.reset_index(drop=True, inplace=True)
training_table_final.set_index('data_prioridade', inplace=True)
training_table_final.head()

,data,prioridade,qtd_inc,feriado,dia_semana,dia_util,taxa_stat_si_diario,taxa_stat_ea_diario,taxa_stat_e_diario,taxa_mon_diario,ics_outliers_ativos,ics_inativos
data_prioridade,,,,,,,,,,,,
2023-01-02_p1,2023-01-02,1,0,0,0,1,NaN,NaN,NaN,NaN,NaN,NaN
2023-01-02_p2,2023-01-02,2,0,0,0,1,NaN,NaN,NaN,NaN,NaN,NaN
2023-01-02_p3,2023-01-02,3,1,0,0,1,NaN,NaN,NaN,NaN,NaN,NaN
2023-01-02_p4,2023-01-02,4,0,0,0,1,NaN,NaN,NaN,NaN,NaN,NaN
2023-01-02_p5,2023-01-02,5,0,0,0,1,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Removendo o primeiro dia do treinamento
# filtrando data > 2023-01-02
training_table_version1 = training_table_final[training_table_final['data'] > '2023-01-02']
training_table_version1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5470 entries, 2023-01-03_p1 to 2025-12-31_p5
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   data                 5470 non-null   datetime64[ns]
 1   prioridade           5470 non-null   int64         
 2   qtd_inc              5470 non-null   int64         
 3   feriado              5470 non-null   int64         
 4   dia_semana           5470 non-null   int64         
 5   dia_util             5470 non-null   int64         
 6   taxa_stat_si_diario  5470 non-null   float64       
 7   taxa_stat_ea_diario  5470 non-null   float64       
 8   taxa_stat_e_diario   5470 non-null   float64       
 9   taxa_mon_diario      5470 non-null   float64       
 10  ics_outliers_ativos  5470 non-null   float64       
 11  ics_inativos         5470 non-null   float64       
dtypes: datetime64[ns](1), float64(6), int64(5)
memory usage: 555.5+ KB


### Training ARIMA